# Phase 1 EDA — NYC listings

Look-only notebook. No cleaning/filtering logic lives here — that's Phase 2, in `src/pricelens/data/clean.py`, tested. This notebook only calls into `src/pricelens/` and plots what comes back.

Covers the 5 items from `PROJECT_PLAN.md` Phase 1: price distribution (raw + log), missingness, geo scatter, the `minimum_nights` cliff, and host concentration (which motivates `GroupKFold` in Phase 3).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from pricelens.config import load_city_config
from pricelens.data.clean import parse_price
from pricelens.data.load import load_listings

plt.rcParams["figure.dpi"] = 100

In [ ]:
CFG = load_city_config(Path("../configs/city/nyc.yaml"))
LISTINGS_PATH = Path("../data/raw") / CFG.city / CFG.scrape_date / "listings.csv.gz"

listings = load_listings(LISTINGS_PATH, CFG)
print(listings.shape)
listings.dtypes.value_counts()

## 1. Price distribution (raw + log)

`price` is still the raw `"$113.97"` string at this point — `parse_price` is the one piece of Phase 2 pulled forward (tested in `tests/test_clean.py`), since you can't histogram a string.

In [ ]:
price = parse_price(listings["price"]).astype("float64")
log_price = np.log1p(price)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(price.dropna(), bins=80)
axes[0].set_title("price ($)")
axes[0].set_xlabel("price")

axes[1].hist(log_price.dropna(), bins=80)
axes[1].set_title("log1p(price)")
axes[1].set_xlabel("log1p(price)")
fig.tight_layout()

print("missing price:", int(price.isna().sum()), "/", len(price))
price.describe()

## 2. Missingness

In [ ]:
null_frac = listings.isna().mean().sort_values(ascending=False)
top_missing = null_frac[null_frac > 0].head(25)

fig, ax = plt.subplots(figsize=(8, 8))
top_missing.iloc[::-1].plot.barh(ax=ax)
ax.set_xlabel("fraction missing")
ax.set_title("Top columns by missingness")
fig.tight_layout()

top_missing

## 3. Geo scatter

In [ ]:
fig, ax = plt.subplots(figsize=(7, 8))
sc = ax.scatter(
    listings["longitude"],
    listings["latitude"],
    c=log_price,
    cmap="viridis",
    s=3,
    alpha=0.4,
)
ax.set_xlim(CFG.bbox.lon_min, CFG.bbox.lon_max)
ax.set_ylim(CFG.bbox.lat_min, CFG.bbox.lat_max)
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.set_title("Listings by location, coloured by log1p(price)")
fig.colorbar(sc, ax=ax, label="log1p(price)")

## 4. `minimum_nights` cliff

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
listings["minimum_nights"].clip(upper=60).plot.hist(bins=61, ax=ax)
ax.axvline(30, color="red", linestyle="--", label="30-night threshold")
ax.set_xlabel("minimum_nights (clipped at 60)")
ax.legend()

listings["minimum_nights"].value_counts().sort_index().loc[25:35]

## 5. Host concentration

This is the number that motivates `GroupKFold` on `host_id` in Phase 3 — random K-fold lets near-duplicate listings from the same host land on both sides of a split.

In [ ]:
host_counts = listings["host_id"].value_counts()

fig, ax = plt.subplots(figsize=(8, 4))
host_counts.value_counts().sort_index().head(20).plot.bar(ax=ax)
ax.set_xlabel("listings per host")
ax.set_ylabel("number of hosts")
ax.set_title("How many hosts have N listings")

multi_listing_share = listings["host_id"].isin(host_counts[host_counts > 1].index).mean()
print(f"{multi_listing_share:.1%} of listings belong to a host with 2+ listings")
host_counts.head(10)

## Data-quality notes

**Phase 1 gate:** be able to state the top 5 data-quality problems in this snapshot out loud, then write them into `reports/model_card.md § Data`.

Questions to work from (write your own findings, not just answers to these):

- How far out does the price distribution's right tail go — do the placeholder/outlier prices look like they'll get caught by the p99.5-within-`room_type` cap planned for Phase 2?
- Which columns are missing often enough that a `_was_missing` indicator (not deletion, not mean-imputation) is the right call?
- Does the geo scatter show any points outside where NYC listings should actually be?
- How sharp is the `minimum_nights = 30` spike, and roughly how many listings will the `<= 30` filter (Phase 2) drop?
- What fraction of listings come from multi-listing hosts — does it line up with the ~30% figure in `TECHNICAL_DESIGN.md §5`?

*(fill in below)*

1. 
2. 
3. 
4. 
5. 